In [ ]:
"""
即時剪貼簿去識別化 / 還原工具 (Qwen2.5-Coder 分段 + 正則 + Microsoft Presidio)
- 複製文字後，先用本地 Qwen2.5-Coder 模型把內容切成 code / text / question 段落
  - code 段落：只跑程式碼類正則 (API key、JWT、電腦路徑)
  - text/question 段落：跑個資類正則 (電話、信用卡等) + Presidio 模型
- 複製已含 <ENTITY_n> 標籤的文字 -> 不掃描，直接查表還原成原始敏感資訊並貼回剪貼簿
- 標籤對應存於 clipboard_map.txt (格式: 標籤\t原始內容)

安裝: pip install presidio-analyzer presidio-anonymizer pyperclip pywin32 transformers torch accelerate
      python -m spacy download en_core_web_lg
      (首次執行會自動下載 Qwen/Qwen2.5-Coder-1.5B-Instruct)
"""
import re
import time
import json
import pyperclip
from dataclasses import dataclass
from typing import Callable, Optional
from presidio_analyzer import AnalyzerEngine
from transformers import AutoModelForCausalLM, AutoTokenizer

MAP_FILE = "clipboard_map.txt"
TAG_RE = re.compile(r"<([A-Z_]+)_(\d+)>")

analyzer = AnalyzerEngine()

# ---------- 初步分類模型 (Qwen2.5-Coder)：先把文字切成 code/text/question 段落，決定要跑哪一套正則 ----------
# 只要改這一個變數就能切換行為，不用動其他程式碼：
#   "auto" -> 呼叫 Qwen 模型分類 code/text 段落 (效果最好，但較慢、吃記憶體)
#   "code" -> 略過分類，全部當「程式碼」處理，只跑 CODE_FILTERS
#   "text" -> 略過分類，全部當「一般文字」處理，只跑 TEXT_FILTERS (含 Presidio)
CLASSIFY_MODE = "text"

CLASSIFY_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
classify_tokenizer = None
classify_model = None

if CLASSIFY_MODE == "auto":
    print("正在載入 Qwen2.5-Coder 分類模型...")
    classify_tokenizer = AutoTokenizer.from_pretrained(CLASSIFY_MODEL_NAME)
    try:
        classify_model = AutoModelForCausalLM.from_pretrained(
            CLASSIFY_MODEL_NAME, torch_dtype="auto", device_map="auto"
        )
    except ValueError:
        print("[警告] device_map=auto 記憶體不足以放到 GPU/自動配置，改用 CPU 載入")
        classify_model = AutoModelForCausalLM.from_pretrained(CLASSIFY_MODEL_NAME, torch_dtype="auto")
    print("分類模型載入完成")

CLASSIFY_PROMPT = """請將輸入的文本按照順序切分成 JSON 陣列，類型包含："code"、"text"、"question"。
請直接輸出標準 JSON，格式如下：
{
  "segments": [
    {"segment_type": "text", "content": "..."}
  ]
}

待處理文字：
"""


def classify_segments(text):
    """呼叫本地模型把文字切成段落並標出類型，回傳 [{"segment_type":..., "content":...}, ...]"""
    messages = [
        {"role": "system", "content": "你是一個精準的文本與程式碼結構化解析助手。請僅輸出 JSON 格式。"},
        {"role": "user", "content": CLASSIFY_PROMPT + text},
    ]
    chat_text = classify_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = classify_tokenizer([chat_text], return_tensors="pt").to(classify_model.device)
    output_ids = classify_model.generate(**inputs, max_new_tokens=1024, temperature=0.1)
    resp = classify_tokenizer.batch_decode(
        [o[len(i):] for i, o in zip(inputs.input_ids, output_ids)], skip_special_tokens=True
    )[0]
    try:
        data = json.loads(resp[resp.find("{"): resp.rfind("}") + 1])
        segments = data.get("segments", [])
    except Exception:
        segments = [{"segment_type": "text", "content": text}]  # 解析失敗就整段當一般文字處理，確保不漏掃
    print(f"[分類完成] 共切出 {len(segments)} 個段落")
    return segments


def locate_segments(text, segments):
    """模型回傳的是切出來的內容，這裡把每段內容對應回原始文字的座標 (start, end, segment_type)"""
    cursor = 0
    located = []
    for seg in segments:
        content = seg.get("content", "")
        if not content:
            continue
        idx = text.find(content, cursor)
        if idx == -1:
            idx = text.find(content)  # 順序找不到就整篇再找一次
        if idx == -1:
            continue  # 對不到位置就跳過這段，不影響其他段落
        located.append((idx, idx + len(content), seg.get("segment_type", "text")))
        cursor = idx + len(content)
    return located


def luhn_valid(value: str) -> bool:
    """信用卡卡號 Luhn 演算法驗證，供 RULES 的 validator 使用"""
    digits = [int(d) for d in value if d.isdigit()]
    if not (13 <= len(digits) <= 19):
        return False
    checksum, parity = 0, len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        checksum += d
    return checksum % 10 == 0


FILE_EXTENSIONS = (
    r"txt|docx?|xlsx?|pptx?|pdf|png|jpe?g|gif|bmp|svg|csv|zip|rar|7z|tar|gz|"
    r"mp4|mp3|wav|py|ipynb|js|ts|jsx|tsx|json|xml|ya?ml|log|ini|cfg|conf|env|sql|html?|css|sh|bat|exe|dll"
)


def quoted_sensitive_valid(value: str) -> bool:
    """給 QUOTED_SECRET 用：引號裡的內容要像網址/檔名/密碼才算數，避免把一般引言也遮掉"""
    if re.match(r"^https?://", value, re.IGNORECASE):
        return True
    if re.search(rf"\.(?:{FILE_EXTENSIONS})$", value, re.IGNORECASE):
        return True
    # 密碼風格：不含空白、長度 4~64、同時有英文字母與數字
    return (" " not in value and 4 <= len(value) <= 64
            and re.search(r"[A-Za-z]", value) and re.search(r"[0-9]", value))


# ========== 正則快速比對規則區：想增加/刪除規則，只要在下面 RULES 這個 list 新增/刪除一行 Rule(...) ==========
@dataclass
class Rule:
    name: str                                          # 標籤用的類別名稱 (etype)
    pattern: str                                        # 正則表達式字串
    flags: int = 0                                      # re flags，例如 re.IGNORECASE
    validator: Optional[Callable[[str], bool]] = None   # 二次驗證函式（可選，降低誤報）
    category: str = "text"                              # "code": 只在一般文字段落"不會"額外重複；純標記用途，仍會出現在程式碼段落
    code_only: bool = False                             # True: 只在程式碼段落跑 (例如 FILE_PATH，一般文字不太會出現)

RULES = [
    # -- 只在程式碼段落才需要的：檔案路徑 (一般文字很少出現這種格式) --
    Rule("FILE_PATH", r"(?:[A-Za-z]:\\(?:[^\\/:*?\"<>|\r\n]+\\)*[^\\/:*?\"<>|\r\n]+)|(?:/(?:[^/\s]+/)+[^/\s]+)", category="code", code_only=True),

    # -- 程式碼、文字都要跑的：API key、JWT --
    Rule("JWT", r"\beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\b", category="code"),
    Rule("API_KEY_OPENAI", r"\bsk-(?:proj-|ant-)?[A-Za-z0-9_-]{20,}\b", category="code"),
    Rule("API_KEY_AWS", r"\b(?:AKIA|ASIA)[0-9A-Z]{16}\b", category="code"),
    Rule("API_KEY_GITHUB", r"\b(?:ghp|gho|ghu|ghs|ghr|github_pat)_[A-Za-z0-9_]{20,}\b", category="code"),
    Rule("API_KEY_GOOGLE", r"\bAIza[0-9A-Za-z\-_]{35}\b", category="code"),

    # -- 個資、憑證：一般文字要跑，且程式碼段落也一起跑（程式碼裡也常混著這些） --
    Rule("TW_PHONE", r"\b09\d{2}[- ]?\d{3}[- ]?\d{3}\b", category="text"),
    Rule("CREDIT_CARD", r"\b(?:\d[ -]?){13,19}\b", validator=luhn_valid, category="text"),
    Rule("URL", r"\bhttps?://[^\s\"'<>]+", category="text"),   # 整個網址一起遮，內含的帳密/token 一併蓋掉
    Rule("FILE_NAME", rf"\b[\w\-. ]+?\.(?:{FILE_EXTENSIONS})\b", flags=re.IGNORECASE, category="text"),
    # 中長度帳密風格字串：8~29 字、同時含英文字母與數字 (沒引號時門檻較高，降低誤判)
    Rule("CREDENTIAL_LIKE",
         r"\b(?=[A-Za-z0-9]{8,29}\b)(?=[A-Za-z0-9]*[A-Za-z])(?=[A-Za-z0-9]*[0-9])[A-Za-z0-9]{8,29}\b",
         category="text"),
    # 通用長亂碼字串（如 Unsplash access/secret key）：長度>=30、須同時含英文字母與數字
    Rule("GENERIC_SECRET", r"\b(?=[A-Za-z0-9_-]{30,}\b)(?=[A-Za-z0-9_-]*[A-Za-z])(?=[A-Za-z0-9_-]*[0-9])[A-Za-z0-9_-]{30,}\b", category="text"),
    # 被引號 "" '' “” ‘’ 標起來的內容：只要符合網址/檔名/密碼樣式就算，門檻可以比沒引號時更寬鬆(短到 4 字)
    Rule("QUOTED_SECRET", r'["\'“”‘’]([^"\'“”‘’\r\n]{4,100})["\'“”‘’]', validator=quoted_sensitive_valid, category="text"),
    # 範例：新增自己的規則就照這個格式加一行，例如：
    # Rule("TW_ID", r"\b[A-Z][12]\d{8}\b", validator=tw_id_valid, category="text"),
]

# 程式碼段落：code_only 規則 + 全部一般文字規則（個資也要在程式碼裡一起抓）
_CODE_RULES = [(r.name, re.compile(r.pattern, r.flags), r.validator) for r in RULES if r.category == "code" or r.code_only]
_TEXT_RULES = [(r.name, re.compile(r.pattern, r.flags), r.validator) for r in RULES if r.category == "text" and not r.code_only]
_CODE_RULES += _TEXT_RULES   # 程式碼段落也套用所有一般文字(個資)規則


def _run_rules(text, compiled_rules):
    """若 pattern 裡有擷取群組 (如 QUOTED_SECRET 的引號內容)，用群組的座標；否則用整個 match 的座標"""
    matches = []
    for name, pattern, validator in compiled_rules:
        for m in pattern.finditer(text):
            if pattern.groups >= 1:
                value, start, end = m.group(1), m.start(1), m.end(1)
            else:
                value, start, end = m.group(0), m.start(), m.end()
            if validator is None or validator(value):
                matches.append((start, end, name))
    return matches


def code_regex_filter(text):
    """程式碼段落用：API key / JWT / 檔案路徑 + 全部個資規則一起跑"""
    return _run_rules(text, _CODE_RULES)


def text_regex_filter(text):
    """一般文字段落用：電話、信用卡、URL、檔名、帳密風格字串等個資規則"""
    return _run_rules(text, _TEXT_RULES)


# ---------- 過濾器區：想加/換偵測方式，寫一個回傳 [(start,end,entity_type),...] 的函式，放進對應的 list 即可 ----------
def presidio_filter(text):
    return [(r.start, r.end, r.entity_type) for r in analyzer.analyze(text=text, language="en")]


CODE_FILTERS = [code_regex_filter]                    # 判斷為程式碼的段落只跑這些
TEXT_FILTERS = [text_regex_filter, presidio_filter]   # 判斷為文字/問題的段落跑這些 (含 Presidio 模型)
ALL_FILTERS = CODE_FILTERS + TEXT_FILTERS             # 分類失敗時的保底：全部規則都跑一遍


# ---------- 對應表讀寫 ----------
def load_map():
    mapping = {}
    try:
        with open(MAP_FILE, encoding="utf-8") as f:
            for line in f:
                tag, _, val = line.rstrip("\n").partition("\t")
                if tag:
                    mapping[tag] = val
    except FileNotFoundError:
        pass
    return mapping


def save_map(mapping):
    with open(MAP_FILE, "w", encoding="utf-8") as f:
        for tag, val in mapping.items():
            f.write(f"{tag}\t{val}\n")


def merge_overlaps(matches):
    """多個過濾器/規則可能比對到重疊區間，若直接替換會切壞字串，
    這裡依起始位置排序、同起點取較長者，彼此重疊的區間只保留先出現(較長)的那個"""
    matches = sorted(matches, key=lambda m: (m[0], -(m[1] - m[0])))
    merged, last_end = [], -1
    for start, end, etype in matches:
        if start >= last_end:
            merged.append((start, end, etype))
            last_end = end
    return merged


# ---------- 遮蔽 / 還原 ----------
def anonymize(text, mapping):
    if CLASSIFY_MODE == "code":
        segments = [(0, len(text), "code")]
    elif CLASSIFY_MODE == "text":
        segments = [(0, len(text), "text")]
    else:  # "auto" -> 真的呼叫模型分類
        segments = locate_segments(text, classify_segments(text))
        if not segments:
            segments = [(0, len(text), None)]  # 分類失敗的保底：整段都跑，type=None 觸發 ALL_FILTERS

    matches = []
    for seg_start, seg_end, seg_type in segments:
        content = text[seg_start:seg_end]
        flts = CODE_FILTERS if seg_type == "code" else TEXT_FILTERS if seg_type else ALL_FILTERS
        for flt in flts:
            matches += [(seg_start + s, seg_start + e, etype) for s, e, etype in flt(content)]

    if not matches:
        return None
    matches = merge_overlaps(matches)

    reverse = {v: k for k, v in mapping.items()}
    counters = {}
    for tag in mapping:
        t, i = TAG_RE.match(tag).groups()
        counters[t] = max(counters.get(t, 0), int(i))

    out = text
    for start, end, etype in sorted(matches, key=lambda m: m[0], reverse=True):
        val = text[start:end]
        tag = reverse.get(val)
        if not tag:
            counters[etype] = counters.get(etype, 0) + 1
            tag = f"<{etype}_{counters[etype]}>"
            mapping[tag] = val
            reverse[val] = tag
        out = out[:start] + tag + out[end:]

    save_map(mapping)
    return out


def deanonymize(text, mapping):
    return TAG_RE.sub(lambda m: mapping.get(m.group(0), m.group(0)), text)


# ---------- 主迴圈 ----------
def main():
    mapping = load_map()
    last = ""
    print("監控剪貼簿中... (Ctrl+C 結束)")
    while True:
        try:
            text = pyperclip.paste()
        except Exception:
            text = ""

        if text and text != last:
            print("[偵測到複製] 開始處理剪貼簿內容...")
            new_text = deanonymize(text, mapping) if TAG_RE.search(text) else anonymize(text, mapping)
            if new_text and new_text != text:
                pyperclip.copy(new_text)
                text = new_text
                print("[處理完成] 已更新剪貼簿內容")
            last = text

        time.sleep(0.5)


if __name__ == "__main__":
    main()

: 